Copyright 2026 DataRobot, Inc. and its affiliates.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

In [ ]:
from jointfm_client import bootstrap_notebook

bootstrap_notebook(add_src_root=True)

# Log Probability
The other return modes ask what the model expects. `log_prob` asks the opposite: how plausible does the model find future values you already hold? A backtest is the natural case. Of the 100 positive float daily observations of equity index level, 10-year Treasury yield, one EUR/USD FX rate, USD portfolio NAV, and realized volatility, hold out the last 10, forecast from the first 90, and score the held-out rows.

`client.forecast_log_prob(...)` takes the observed values as `query_rows`: one row per entry of `query_times`, with a finite value for every declared column. The score of a row is the model's log density of the whole row under its joint at that position, so it covers every declared column, features included; `requested_columns`, if passed at all, must list every declared column in declared order. The result's `log_prob` block holds one score per query time in `values`, their sum in `total`, their average in `mean`, and the negative log-likelihood views `nll_values`, `nll_total`, and `nll_mean`. `to_pandas_tidy()` returns one row per query time.

In [ ]:
import math
from pathlib import Path

import pandas as pd

from jointfm_client import JointFMClient, plan_forecast_columns

HISTORY_PATH = Path("notebooks/history.csv")
FEATURE_COLUMNS = ["equity_index_level", "treasury_10y_yield", "eur_usd_rate"]
TARGET_COLUMNS = ["portfolio_nav", "realized_volatility"]
OBSERVED_STEPS = 100
HELD_OUT_STEPS = 10
EXPECTED_COLUMNS = FEATURE_COLUMNS + TARGET_COLUMNS

observed = pd.read_csv(HISTORY_PATH, dtype=float)
if list(observed.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        f"Expected columns {EXPECTED_COLUMNS!r}, got {list(observed.columns)!r}"
    )
if len(observed) != OBSERVED_STEPS:
    raise ValueError(f"Expected {OBSERVED_STEPS} observed rows, got {len(observed)}")
history = observed.iloc[:-HELD_OUT_STEPS].reset_index(drop=True)
held_out = observed.iloc[-HELD_OUT_STEPS:].reset_index(drop=True)
QUERY_TIMES = list(range(len(history), OBSERVED_STEPS))

client = JointFMClient.from_env()
instance_ids = [
    entry.instance_id for entry in client.health_instances(cache=True).instances
]
print(f"JointFM instance(s): {instance_ids}")
plan = plan_forecast_columns(
    health=client.health(cache=True),
    feature_columns=FEATURE_COLUMNS,
    target_columns=TARGET_COLUMNS,
    history_length=len(history),
    query_times_length=len(QUERY_TIMES),
)
scored = client.forecast_log_prob(
    history,
    query_times=QUERY_TIMES,
    query_rows=held_out,
    columns=plan.columns,
)
per_step = scored.to_pandas_tidy()
if len(per_step) != HELD_OUT_STEPS:
    raise ValueError(f"Expected {HELD_OUT_STEPS} scored rows, got {len(per_step)}")
print("log density of the held-out block:", scored.log_prob.total)
per_step

## Reading a score against an alternative
A log density has no fixed scale of its own: it moves with the units of the columns, since rescaling a column shifts every score by a constant, so a single score says little. Its use is comparison, of one set of rows under two models, or of two candidate outcomes under one model. The difference of two scores is a log-likelihood ratio: a ratio of `r` means the model finds the first candidate `exp(r)` times as plausible as the second.

Below, the realized rows compete against a counterfactual in which portfolio NAV ends every held-out day 5% lower while the market inputs move exactly as they did, a loss the history of those inputs does not explain. A positive ratio at a step means the model prefers what actually happened there.

In [ ]:
STRESS_NAV_FACTOR = 0.95

stressed = held_out.assign(portfolio_nav=held_out["portfolio_nav"] * STRESS_NAV_FACTOR)
stressed_scored = client.forecast_log_prob(
    history,
    query_times=QUERY_TIMES,
    query_rows=stressed,
    columns=plan.columns,
)
comparison = pd.DataFrame(
    {
        "query_time": QUERY_TIMES,
        "realized": scored.log_prob.values,
        "stressed": stressed_scored.log_prob.values,
    }
)
comparison["log_likelihood_ratio"] = comparison["realized"] - comparison["stressed"]
block_ratio = scored.log_prob.total - stressed_scored.log_prob.total
realized_days = int((comparison["log_likelihood_ratio"] > 0).sum())
print(f"log-likelihood ratio of the block, realized over stressed: {block_ratio:.1f}")
print(
    "The model finds the realized block about "
    f"10^{block_ratio / math.log(10):.0f} times as plausible as the stressed one,"
)
print(
    f"and prefers what actually happened on {realized_days} of {HELD_OUT_STEPS} days."
)
print("Each day is scored on its own, so the block ratio adds up daily surprises.")
comparison